# CapIQ Credit Risk — Executive Summary (2026 Outlook)

**Model**: XGBoost (Optuna-tuned)  
**Targets**: 
- 12-month horizon (Dec 2025 – Dec 2026)
- 5-year horizon (Dec 2025 – Dec 2030)  
**Data**: Clean time-linked Compustat + CRSP + delist signals (corrected labeling)

## Data & Modeling Summary

- **Linked master panel**: 209,498 firm-years (21,135 unique gvkeys)
- **12-month model** (default_in_next_12m): Dec 2025 – Dec 2026 risk window
- **5-year model** (default_in_next_5y): Dec 2025 – Dec 2030 risk window (2026 data only + alive companies)
- **Final features**: 34 numeric credit-risk ratios (leverage, profitability, liquidity, coverage, size, efficiency)
- **Best hyperparameters** (Optuna): n_estimators=547, max_depth=10, learning_rate≈0.060, subsample≈0.803, colsample_bytree≈0.929
- **2026 predictions** generated on the latest fiscal observation (standardized risk windows: Dec 2025–Dec 2026 for 12m, Dec 2025–Dec 2030 for 5y). All outputs organized under `outputs/12m_model/` and `outputs/5y_model/`.

In [3]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
OUT_12M = PROJECT_ROOT / 'outputs' / '12m_model'
OUT_5Y  = PROJECT_ROOT / 'outputs' / '5y_model'

# Load 12-month model results
top_12m = pd.read_csv(OUT_12M / 'top_2026_companies.csv')
print("=== 12-MONTH MODEL (default_in_next_12m) ===\n")
print(top_12m.head(15).to_string(index=False))

# Load 5-year model results (latest 2026 window, alive companies)
top_5y = pd.read_csv(OUT_5Y / 'top_2026_risk_5y_2026data_only_alive.csv')
print("\n\n=== 5-YEAR MODEL (default_in_next_5y, 2026 data only, alive firms) ===\n")
print(top_5y.head(15).to_string(index=False))

# Overlap analysis
common = set(top_12m['conm'].head(20)) & set(top_5y['conm'].head(20))
print(f"\n\nOverlap in top-20 riskiest companies (12m vs 5y): {len(common)} companies")
if common:
    print("Common names:", sorted(common)[:8])


Top 10 Highest 2026 Default Risk Companies


,gvkey,conm,latest_observed_date,risk_horizon,pred_risk_2026
0,37490,LUMINAR TECHNOLOGIES INC,2025-12-31,Dec 2025 → Dec 2026,95.6%
1,345920,HYDROFARM HLDNG GP INC,2025-12-31,Dec 2025 → Dec 2026,95.5%
2,37541,BIOATLA INC,2025-12-31,Dec 2025 → Dec 2026,95.5%
3,50382,ZSPACE INC,2025-12-31,Dec 2025 → Dec 2026,95.0%
4,38804,ZYVERSA THERAPEUTICS INC,2025-12-31,Dec 2025 → Dec 2026,95.0%
5,36519,IM CANNABIS CORP,2025-12-31,Dec 2025 → Dec 2026,95.0%
6,10235,TRINITY PLACE HOLDINGS INC,2025-12-31,Dec 2025 → Dec 2026,94.7%
7,41686,SNAIL INC,2025-12-31,Dec 2025 → Dec 2026,94.7%
8,62302,SPAR GROUP INC,2025-12-31,Dec 2025 → Dec 2026,94.7%
9,27574,TPI COMPOSITES INC,2025-12-31,Dec 2025 → Dec 2026,94.6%


## 12-Month vs 5-Year Model Comparison

- **12-month model** (`default_in_next_12m`): Standardized risk window = **Dec 2025 - Dec 2026 (12 months from last fiscal year)**. Most sensitive to near-term distress.
- **5-year model** (`default_in_next_5y`): Standardized risk window = **Dec 2025 - Dec 2030 (5 years from last fiscal year)**. Captures longer-term solvency risk. Uses only 2026-dated observations of companies that have not yet defaulted (`bankrupt_delist == 0`).
- Use both lists: companies high on both are structural high-risk names.
- Companies high only on 5y may be currently healthy but have deteriorating fundamentals over a longer horizon.

## What Each Important Feature Means (for SHAP Interpretation)

All SHAP plots and waterfalls below use these plain-English definitions:

In [4]:
import pandas as pd
from pathlib import Path
defs = pd.read_csv((Path.cwd().parent / 'outputs' / 'feature_definitions.csv') if (Path.cwd().parent / 'outputs' / 'feature_definitions.csv').exists() else (Path.cwd().parent / 'outputs' / '12m_model' / 'feature_definitions.csv'))
display(defs.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))


feature,business_meaning
leverage,Total Debt / Total Assets — core leverage risk
debt_at,Total Debt / Total Assets
de_ratio,Debt / Equity — gearing
ebitda_margin,EBITDA / Sales — operating profitability (higher = safer)
roa,Return on Assets
roe,Return on Equity
bm,Book-to-Market (high often signals distress)
curr_ratio,Current Ratio — short-term liquidity
quick_ratio,Quick Ratio — liquidity ex-inventory
cash_ratio,Cash Ratio — most conservative liquidity


## Key SHAP Insights (Global Feature Impact)

- **High leverage** (`leverage`, `debt_ebitda`, `de_ratio`) is the strongest driver of elevated default risk.
- **Weak profitability** (`ebitda_margin`, `roa`) dramatically increases predicted probability.
- **Poor liquidity** (`curr_ratio`, `cash_ratio`, `quick_ratio`) and low **interest coverage** (`intcov_ratio`) push firms into the high-risk tail.
- **Small firm size** (`log_at`) is a consistent risk factor (smaller firms have fewer resources to weather distress).

See `outputs/12m_model/shap_beeswarm.png`, `shap_bar.png`, and `shap_global_summary.png` for the full picture.

## Individual Company Explanations (Waterfall Plots)

For each of the top-10 riskiest companies we generated a SHAP waterfall showing exactly which features pushed their predicted default probability up or down.

Files: `waterfall_rank1_....png` to `waterfall_rank10_....png` (and individual company-named files) are in `outputs/12m_model/`.

These plots are the most actionable output — they tell a credit analyst or investor precisely why a particular company is flagged as high risk in 2026.

## Recommendation

Monitor the top 10–20 names on both the 12-month (Dec 2025–Dec 2026) and 5-year (Dec 2025–Dec 2030) risk lists closely. Focus especially on companies where **leverage + weak EBITDA margin + low liquidity** combine — these are the classic precursors to distress.

The model (and the corrected labeling) now gives trustworthy, non-leaky forward-looking probabilities.